Question 1:

You are trying to use the T5 model to perform English-to-French translation. The model expects tokenized input.


In [5]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

text = "translate English to French: How are you?"

# Tokenize input
inputs = tokenizer(text, return_tensors="pt")

# Generate
outputs = model.generate(**inputs)

# Decode
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Comment êtes-vous?



Question 2:

You wrote a RAG-like prompt flow but skipped the vector retrieval step. Complete it.


In [6]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import numpy as np

# Step 1: embedder
embedder = SentenceTransformer("all-MiniLM-L6-v2")

documents = [
    "Paris is the capital of France.",
    "Berlin is the capital of Germany."
]

doc_embeddings = embedder.encode(documents)

# Step 2: retrieval
query = "What is the capital of France?"
query_embedding = embedder.encode([query])

scores = np.dot(doc_embeddings, query_embedding.T).squeeze()
best_doc = documents[scores.argmax()]

# Step 3: generation
generator = pipeline("text-generation", model="distilgpt2")

prompt = f"Context: {best_doc}\nQuestion: {query}\nAnswer:"
output = generator(prompt, max_length=50)

print(output[0]["generated_text"])


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Context: Paris is the capital of France.
Question: What is the capital of France?
Answer: The capital is France, the place that your brother and sister are. In this context, it is France.
This question was raised specifically



Question 3:

You are trying to pass plain text to a model without tokenizing. Fix it to generate correct output using the Hugging Face T5 model.


In [21]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

text = "translate to french: Have a wonderful day."

# FIX: tokenize first
inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(**inputs)
translate = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(translate)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Vous avez un beau jour.



Question 4:

What is the correct code to summarize a paragraph using a pre-trained BART model from Hugging Face?



In [22]:
from transformers import BartTokenizer, BartForConditionalGeneration

model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

text = """
Artificial intelligence is transforming industries by enabling machines
to learn from data and perform tasks that typically require human intelligence.
"""

inputs = tokenizer(text, max_length=1024, return_tensors="pt", truncation=True)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    min_length=20,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)


Artificial intelligence is transforming industries by enabling machines to learn from data and perform tasks that typically require human intelligence.


Question 5:

You want to generate a small sentence using a GenAI model. Use the Hugging Face pipeline to do it with a tiny model. What’s the correct code?

In [39]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

output = generator("AI is changing", max_length=20)
print(output[0]["generated_text"])


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


AI is changing our culture over the last 40 years, with new research from the Massachusetts Institute of Technology
